In [14]:
import anatomist.api as ana
from soma.qt_gui.qtThread import QtThreadCall
from soma.qt_gui.qt_backend import Qt

a = ana.Anatomist()

from soma import aims
from scipy import ndimage
import numpy as np
import glob
import os

In [3]:
subject = "197550"
side = "R"

sources = glob.glob(f'/neurospin/dico/data/deep_folding/current/datasets/hcp/crops/2mm/*')
file_src = f'/neurospin/dico/data/deep_folding/current/datasets/hcp/skeletons/2mm/{side}/{side}resampled_skeleton_{subject}.nii.gz'

In [20]:
def to_bucket(obj):
    if obj.type() == obj.BUCKET:
        return obj
    avol = a.toAimsObject(obj)
    c = aims.Converter(intype=avol, outtype=aims.BucketMap_VOID)
    abck = c(avol)
    bck = a.toAObject(abck)
    bck.releaseAppRef()
    return bck

def crop_mask(file_src, file_cropped, mask):
    """Crops according to mask"""
    volume = aims.read(file_src)
    print(np.count_nonzero(volume.np))
    if mask:
        mask = aims.read(mask)
        arr = volume.np
        arr_mask = np.asarray(mask)
        arr[arr_mask == 0] = 0
        print(np.count_nonzero(volume.np))
    aims.write(volume, file_cropped)

In [ ]:
"""
bv bash
cd /volatile/ad279118/deep_folding
. venv/bin/activate
pip install -e .
cd /volatile/ad279118/deep_folding/deep_folding/brainvisa/utils
python3 convert_volume_to_bucket.py -s '/volatile/ad279118/Figures_report/197550' -t '/volatile/ad279118/Figures_report/197550'
"""

In [48]:
w = a.createWindow("3D")
w2 = a.createWindow("3D")
dic_windows = {}
list_regions = []

In [49]:
# to plot the white mesh of the same subject
dic_windows[f'source_{subject}'] = a.loadObject(file_src)
dic_windows[f'source_{subject}'].loadReferentialFromHeader()
dic_windows[f'fusion_{subject}'] = a.fusionObjects(objects=[dic_windows[f'source_{subject}']], method='VolumeRenderingFusionMethod')
w.addObjects(dic_windows[f'fusion_{subject}'])

nifti transfo: 1


ATransformSet::unregisterObserver: ref 0x565985e7e6e0 not found


no position could be read at 664, 531
no position could be read at 793, 523
no position could be read at 720, 532
no position could be read at 701, 544
no position could be read at 752, 528
no position could be read at 649, 479
no position could be read at 713, 439
no position could be read at 687, 468
no position could be read at 707, 461
no position could be read at 648, 483
no position could be read at 763, 488
no position could be read at 713, 567
no position could be read at 730, 530
no position could be read at 735, 527
no position could be read at 786, 546
no position could be read at 736, 547
no position could be read at 758, 549
no position could be read at 761, 555
no position could be read at 748, 549
no position could be read at 727, 544
no position could be read at 705, 558
no position could be read at 732, 551
no position could be read at 698, 573
no position could be read at 762, 576
no position could be read at 733, 588
no position could be read at 719, 492
no position 

In [50]:
for source in sources:
    region = source.replace('/neurospin/dico/data/deep_folding/current/datasets/hcp/crops/2mm/', '')
    if region in ['CENTRAL']:
        continue
    list_regions.append(region)
    mask_path = f'{source}/mask/{side}mask_skeleton.nii.gz'
    #file_cropped = f'/volatile/ad279118/Figures_report/{subject}/{subject}_{region}_{side}_cropped_skeleton.nii.gz'
    #crop_mask(file_src, file_cropped, mask_path)
    #dic_windows[f'vol_{region}'] = aims.read(file_cropped)
    #dic_windows[f'a_obj_{region}'] = a.toAObject(dic_windows[f'vol_{region}'])
    #dic_windows[f'fusion_{region}'] = a.fusionObjects(objects=[dic_windows[f'a_obj_{region}']], method='VolumeRenderingFusionMethod')
    #w.addObjects(dic_windows[f'fusion_{region}'])

    #
    path_to_bck = f"/volatile/ad279118/Figures_report/197550/{subject}_{region}_{side}_cropped_skeleton.bck"
    dic_windows[f'bcks_{region}'] = a.loadObject(path_to_bck)
    dic_windows[f'bcks_{region}'].loadReferentialFromHeader()
    w2.addObjects(dic_windows[f'bcks_{region}'])

no position could be read at 1410, 770
Position : 32.9276, 121.638, 88.316, 0
no position could be read at 856, 582
Position : 48.4509, 158.403, 101.124, 0
Position : 89.8644, 182.072, 103.076, 0
Position : 93.0188, 162.89, 103.076, 0
Position : 47.0154, 137.157, 86.0157, 0
Position : 27.0154, 136.414, 91.0737, 0
Position : 35.0153, 114.587, 88.5656, 0
Position : 97.0153, 88.9631, 61.7425, 0
Position : 79.0153, 74.0446, 61.5216, 0
Position : 38.9656, 65.4659, 92.3288, 0


: 

In [10]:
def compute_bbox_mask(arr):

    # Gets location of bounding box as slices
    objects_in_image = ndimage.find_objects(arr)
    print(f"ndimage.find_objects(arr) = {objects_in_image}")
    if not objects_in_image:
        raise ValueError("There are only 0s in array!!!")

    loc = objects_in_image[0]
    bbmin = []
    bbmax = []

    for slicing in loc:
        bbmin.append(slicing.start)
        bbmax.append(slicing.stop)

    return np.array(bbmin), np.array(bbmax)

def initialize_empty_volume(path):
    # Read the original to get correct dims + header
    ref = aims.read(path)
    ref_arr = np.asarray(ref)
    
    # Create a new empty volume with the same geometry
    full = aims.Volume(ref_arr.shape, dtype='float32')
    full_arr = np.asarray(full)
    full_arr[:] = 0  # or NaN if preferred

    return full_arr


def uncrop_volume(full_arr, decoded_crop, bbmin, bbmax):
    """
    Places the decoded sub-volume back into the original spatial position.
    Takes voxel-wise max between existing values and decoded_crop.
    """
    try:
        # Extract the region where the crop will be inserted
        region = full_arr[
            bbmin[0]:bbmax[0],
            bbmin[1]:bbmax[1],
            bbmin[2]:bbmax[2]
        ]

        # Compute voxel-wise max
        full_arr[
            bbmin[0]:bbmax[0],
            bbmin[1]:bbmax[1],
            bbmin[2]:bbmax[2]
        ] = np.maximum(region, np.asarray(decoded_crop))

    except Exception as e:
        print("Shape conflict! Check the bbox:", e)
    return full_arr

def build_gradient(pal):
    """Build a gradient palette for Anatomist visualization."""
    gw = ana.cpp.GradientWidget(None, 'gradientwidget', pal.header()['palette_gradients'])
    gw.setHasAlpha(True)
    nc = pal.shape[0]
    rgbp = gw.fillGradient(nc, True)
    rgb = rgbp.data()
    npal = pal.np['v']
    pb = np.frombuffer(rgb, dtype=np.uint8).reshape((nc, 4))
    npal[:, 0, 0, 0, :] = pb
    # Convert BGRA to RGBA
    npal[:, 0, 0, 0, :3] = npal[:, 0, 0, 0, :3][:, ::-1]
    pal.update()

In [33]:
initial_volume = "/neurospin/dico/data/deep_folding/current/datasets/ABCD/crops/2mm/S.C.-sylv./mask/Rmask_skeleton.nii.gz"
init_volume = aims.read(initial_volume)

In [35]:
root_path = "/neurospin/dico/data/deep_folding/current/datasets/ABCD/crops/2mm"
decode_root_path = "/neurospin/dico/adufournet/2025_Champollion_Decoder/runs/Champollion_V1_after_ablation_256"
path_initial_volume = f"{root_path}/S.C.-sylv./mask/Rmask_skeleton.nii.gz"
full_arr = initialize_empty_volume(path_initial_volume)
for region in list_regions:
    if region in ['CINGULATE', 'OCCIPITAL']:
        continue
    print(region)
    # read the mask of the given region
    path_to_mask  = f"{root_path}/{region}/mask/Rmask_skeleton.nii.gz"
    mask_volume = aims.read(path_to_mask)
    mask_arr = np.asarray(mask_volume)
    # compute the box for the given region
    bbmin, bbmax = compute_bbox_mask(mask_arr)
    # get the reconstruction of the given region
    region = region.replace(".", "")
    print(bbmax-bbmin)
    npy_recon = glob.glob(f"{decode_root_path}/*{region}*_right*/reconstruction_best_model/197550_decoded.npy")
    if npy_recon:
        vol_npy = np.load(npy_recon[0]).astype(np.float32)
        print(npy_recon[0])
        vol_npy_shape = vol_npy.shape
        print(vol_npy_shape)
        vol_npy = vol_npy.reshape(list(vol_npy_shape)+[1])
        # place the reconstruction in the global volume
        full_arr = uncrop_volume(full_arr=full_arr, decoded_crop=vol_npy, bbmin=bbmin, bbmax=bbmax)

# write the global reconstruction
file_out = "/volatile/ad279118/Figures_report/global_reconstruction/global_test.nii.gz"
vol_aims = aims.Volume(full_arr.reshape(96, 114, 96))
vol_aims.copyHeaderFrom(init_volume.header())
aims.write(vol_aims, file_out)

F.Coll.-S.Rh.
ndimage.find_objects(arr) = [(slice(18, 52, None), slice(37, 99, None), slice(46, 87, None), slice(0, 1, None))]
[34 62 41  1]
(34, 62, 41, 1)
/neurospin/dico/adufournet/2025_Champollion_Decoder/runs/Champollion_V1_after_ablation_256/8_FColl-SRh_right_bce_0.0005/reconstruction_best_model/197550_decoded.npy
(34, 62, 41)
S.F.median-S.F.pol.tr.-S.F.sup.
ndimage.find_objects(arr) = [(slice(22, 52, None), slice(8, 60, None), slice(15, 70, None), slice(0, 1, None))]
[30 52 55  1]
(30, 52, 55, 1)
/neurospin/dico/adufournet/2025_Champollion_Decoder/runs/Champollion_V1_after_ablation_256/36_SFmedian-SFpoltr-SFsup_right_bce_0.0005/reconstruction_best_model/197550_decoded.npy
(30, 52, 55)
S.F.inf.-BROCA-S.Pe.C.inf.
ndimage.find_objects(arr) = [(slice(13, 39, None), slice(12, 52, None), slice(29, 72, None), slice(0, 1, None))]
[26 40 43  1]
(26, 40, 43, 1)
/neurospin/dico/adufournet/2025_Champollion_Decoder/runs/Champollion_V1_after_ablation_256/26_SFinf-BROCA-SPeCinf_right_bce_0.000

In [36]:
vol = aims.read(file_out)
a_obj = a.toAObject(vol)
fusion = a.fusionObjects(objects=[a_obj], method='VolumeRenderingFusionMethod')
w3 = a.createWindow("3D")
w3.addObjects(fusion)

no position could be read at 406, 515


SetObjectPaletteCommand : warning: palette "CustomGradient" not found


In [29]:
nb_columns = 4
block = a.createWindowsBlock(nb_columns)
dic_windows2 = {}
pal = a.createPalette('VR-palette')
pal.header()['palette_gradients'] = "1;1#0;1;1;0#0.994872;0#0;0;0.635897;0.266667;1;1"
# 0;0;0.74359;1;1;1#0;0;0.692308;1;1;1#0;0.6;1;1#0;0;0.161538;0;0.287179;0.266667;0.5;1;0.697436;0.377778;0.830769;0;1;0
build_gradient(pal)
subjects = [
    104416,
    104820,
    105014,
    105115,
    105216,
    105620,
    105923,
    106016,
]

for subject in subjects:
    file_src = f'/neurospin/dico/data/deep_folding/current/datasets/hcp/skeletons/2mm/{side}/{side}resampled_skeleton_{subject}.nii.gz'
    file_recon = f"/volatile/ad279118/Figures_report/global_reconstruction/{side}_{subject}_decoded.nii.gz"
    # load the input 
    dic_windows2[f'w_init_{subject}'] = a.createWindow('3D', block=block)
    dic_windows2[f'obj_init_{subject}'] = a.loadObject(file_src)
    #dic_windows2[f'buck_init_{subject}'] = to_bucket(dic_windows2[f'obj_init_{subject}'])
    dic_windows2[f'fusion__init_{subject}'] = a.fusionObjects(objects=[dic_windows2[f'obj_init_{subject}']], 
                                                            method='VolumeRenderingFusionMethod')
    dic_windows2[f'w_init_{subject}'].addObjects(dic_windows2[f'fusion__init_{subject}'])

    # load the reconstruction
    dic_windows2[f'w_recon_{subject}'] = a.createWindow('3D', block=block)
    dic_windows2[f'obj_recon_{subject}'] = a.loadObject(file_recon)
    dic_windows2[f'fusion_recon_{subject}'] = a.fusionObjects(objects=[dic_windows2[f'obj_recon_{subject}']], 
                                                                        method='VolumeRenderingFusionMethod')
    dic_windows2[f'fusion_recon_{subject}'].setPalette('VR-palette', 
                                                       minVal=0, 
                                                       maxVal=0.5, 
                                                       absoluteMode=True)
    dic_windows2[f'w_recon_{subject}'].addObjects(dic_windows2[f'fusion_recon_{subject}'])
    #dic_windows2[f'w_init_{subject}'].addObjects(dic_windows2[f'fusion_recon_{subject}'])



no position could be read at 413, 143
no position could be read at 452, 121
Position : -0.857392, 135.288, 95.1503, 0
Position : -0.797561, 124.646, 89.3496, 0
no position could be read at 598, 173
no position could be read at 443, 149
no position could be read at 550, 85
no position could be read at 449, 144
no position could be read at 432, 160
no position could be read at 432, 167
no position could be read at 443, 152
no position could be read at 466, 165
no position could be read at 471, 168
no position could be read at 507, 185
no position could be read at 490, 180
no position could be read at 550, 188
no position could be read at 465, 161
no position could be read at 460, 171
no position could be read at 451, 175
no position could be read at 435, 152
no position could be read at 437, 162
no position could be read at 455, 184
no position could be read at 436, 178


In [30]:
snapshot = True
if snapshot:
    for subject in subjects:
        save_dir = "/volatile/ad279118/Figures_report/global_reconstruction"
        dic_windows2[f'w_recon_{subject}'].setHasCursor(0)
        recon_fname = f"{subject}_{side}_global_reconstruction.png"
        recon_img_path = os.path.join(save_dir, recon_fname)
        dic_windows2[f'w_recon_{subject}'].snapshot(recon_img_path, width=1200, height=900)

        init_fname = f"{subject}_{side}_global_input.png"
        init_img_path = os.path.join(save_dir, init_fname)
        dic_windows2[f'w_init_{subject}'].snapshot(init_img_path, width=1200, height=900)

no position could be read at 722, 176
no position could be read at 346, 123
no position could be read at 406, 100
no position could be read at 390, 95
no position could be read at 464, 172
no position could be read at 480, 164
no position could be read at 560, 139
no position could be read at 469, 189
no position could be read at 321, 113
no position could be read at 409, 129
no position could be read at 368, 186
no position could be read at 397, 155
no position could be read at 408, 154
no position could be read at 478, 66
no position could be read at 432, 109
no position could be read at 442, 110
no position could be read at 442, 123
no position could be read at 404, 133
no position could be read at 417, 128
no position could be read at 365, 149
no position could be read at 376, 162
no position could be read at 560, 160
no position could be read at 425, 176
no position could be read at 516, 186
no position could be read at 444, 156
no position could be read at 666, 100
no position co

In [ ]:
# to plot the white mesh of the same subject
path_to_t1mri = f'/neurospin/dico/data/bv_databases/human/not_labeled/hcp/hcp/{subject}/t1mri/BL'
dic_windows[f'white_{subject}'] = a.loadObject(f'{path_to_t1mri}/default_analysis/segmentation/mesh/{subject}_{side}white.gii')
dic_windows[f'white_{subject}'].loadReferentialFromHeader()